# 2-2절 연습 문제 풀이

이 노트북은 2-2절 연습 문제(2-4 ~ 2-6)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `code_examples/ch02/02-02_example.ipynb`를 참고한다.
- 위에서부터 차례대로 실행한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
# 환경 설정 - 시드 고정 (예제 노트북과 같은 SEED)
import random

import numpy as np
import torch

SEED = 2
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 공통 준비

본문 2-2절의 모델, 손실 함수, 최적화 함수, 학습 루프를 그대로 옮겨 온다.
학습 로그는 문제마다 수백 줄이 되므로 마지막 결과만 돌려주는 조용한 학습 함수를 함께 둔다.

In [2]:
LR = 0.1          # 학습률
EPOCHS = 1000     # 전체 학습 에포크

def sigmoid(z):
    return 1 / (1 + torch.exp(-z))

def model(X, weights, bias):
    return sigmoid(X @ weights + bias)

def criterion(Y_pred, Y_true):
    return ((Y_pred - Y_true) ** 2).mean()

def optimizer(weights, bias, learning_rate):
    with torch.no_grad():
        weights -= learning_rate * weights.grad
        bias -= learning_rate * bias.grad

def train_quiet(X, Y_true, weights, bias, epochs=EPOCHS, learning_rate=LR):
    """[코드 2-5]와 같은 학습 루프. 로그 대신 마지막 손실만 반환한다."""
    for _ in range(epochs):
        loss = criterion(model(X, weights, bias), Y_true)
        loss.backward()
        optimizer(weights, bias, learning_rate)
        weights.grad.zero_()
        bias.grad.zero_()
    return loss.item()

def classify(X, weights, bias):
    """0.5를 문턱값으로 분류 결과를 돌려준다."""
    with torch.no_grad():
        return (model(X, weights, bias) >= 0.5).float()

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y_and = torch.tensor([[0.], [0.], [0.], [1.]])
print(f'입력 {tuple(X.shape)}, AND 정답 {tuple(Y_and.shape)}')

입력 (4, 2), AND 정답 (4, 1)


## 연습 문제 2-4

> `torch.randn()` 함수로 만든 텐서의 요솟값은 평균 0, 표준편차 1인 정규분포를 따르며, 각 요소의 값이 -2~2 사이일
> 확률은 95%이다. 초깃값이 평균 0, 표준편차 10인 정규분포를 따르도록 가중치 초기화를 바꾸면 학습 결과가 어떻게 될까?
> 초기화 방법을 바꿔 서로 다른 초깃값으로 10번 이상 반복 학습해 결과를 확인해 보자.
>
> 힌트: 표준편차가 10인 값으로 초기화한 텐서가 파라미터로 사용하기 적절한지 확인해 볼 필요가 있다.

In [3]:
TRIALS = 10
INIT_STD = 10.    # 초깃값의 표준편차

success = 0
print(f'{"시도":>4} {"학습 후 손실":>12} {"분류 성공":>9}   초깃값 (w1, w2, b)')
print('-' * 60)
for trial in range(1, TRIALS + 1):
    # torch.randn()의 결과에 표준편차를 곱하면 평균 0, 표준편차 INIT_STD인 난수가 된다
    weights = (torch.randn((2, 1)) * INIT_STD).requires_grad_(True)
    bias = (torch.randn((1,)) * INIT_STD).requires_grad_(True)
    init = (weights[0].item(), weights[1].item(), bias.item())
    loss = train_quiet(X, Y_and, weights, bias)
    ok = torch.equal(classify(X, weights, bias), Y_and)
    success += ok
    print(f'{trial:4d} {loss:12.4f} {str(ok):>9}   ({init[0]:7.1f}, {init[1]:7.1f}, {init[2]:7.1f})')
print(f'\n{TRIALS}번 중 분류에 성공한 경우: {success}번')

  시도      학습 후 손실     분류 성공   초깃값 (w1, w2, b)
------------------------------------------------------------
   1       0.0358      True   (    3.9,    -2.2,    -3.2)


   2       0.2590     False   (  -12.1,    10.4,    -6.3)
   3       0.0074      True   (    5.7,     5.4,    -3.9)


   4       0.5001     False   (  -10.4,    13.2,     7.5)
   5       0.2529     False   (  -13.3,   -12.4,    -1.0)


   6       0.2547     False   (   -9.5,     6.2,    -2.4)
   7       0.2527     False   (    0.2,    -4.9,    -0.6)


   8       0.2500     False   (  -15.7,     4.3,   -14.8)
   9       0.2512     False   (   -4.8,     2.5,     1.6)


  10       0.7499     False   (   -1.6,     4.2,    10.0)

10번 중 분류에 성공한 경우: 2번


### 풀이 해설

표준편차 10으로 초기화하면 **열 번 중 두 번만 AND 게이트를 제대로 분류한다.** 나머지 여덟 번은 학습에 실패한다.
실패한 경우의 손실을 보면 0.25, 0.50, 0.75처럼 특정 값에 멈춰 있는데, 이는 모델이 입력과 무관하게
거의 같은 값만 내놓는 상태에 갇혔다는 뜻이다.

원인은 시그모이드 함수의 기울기 소실이다(본문 각주 5). 가중합 z가 10, 20처럼 큰 값이 되면 시그모이드의 출력은
0이나 1에 완전히 붙고, 그 지점의 기울기는 0에 가까워진다. 기울기가 0에 가까우면 아무리 반복해도 파라미터가
거의 움직이지 않는다. 학습이 시작되기도 전에 멈춰 버리는 셈이다.

본문 p8이 "최적값에서 너무 먼 값으로 초기화하면 모델이 수렴하는 데 더 많은 시간이 필요하다"고 말한 것보다
실제 상황은 더 나쁘다. **시간이 더 걸리는 정도가 아니라 아예 수렴하지 못한다.**
파라미터 초기화 방법이 딥러닝에서 따로 연구되는 주제인 이유가 여기에 있다.

### 문제 검토

- **적절성: 적합. 2장에서 가장 값진 문제다.** 본문이 지나가듯 언급한 초기화의 중요성을 열 번의 실험으로
  체감하게 한다. '10번 이상 반복'이라는 지시 덕분에 우연히 성공한 한 번을 보고 잘못 결론 내릴 위험도 막았다.
  실제로 열 번 중 두 번은 성공하므로, 한두 번만 돌려 봤다면 정반대 결론에 이를 수 있었다.
- **[검토] 결과를 판정할 기준이 없다.** "학습 결과가 어떻게 될까"만으로는 무엇을 보고 성공과 실패를 가를지
  알기 어렵다. 손실만 보면 0.25가 큰 값인지 감이 오지 않는다. 분류 결과를 정답과 맞춰 보라고 일러 주면
  판정이 분명해진다.
- **[검토] 힌트가 방향을 덜 가리킨다.** "파라미터로 사용하기 적절한지 확인해 볼 필요가 있다"는 무엇을 확인할지
  모호하다. 기울기 소실이라는 원인을 향하게 하면 독자가 [그림 2-4]와 각주 5로 되돌아가게 된다.

**윤문안**

> **2-4**. `torch.randn()` 함수로 만든 텐서의 요솟값은 평균 0, 표준편차 1인 정규분포를 따르며, 각 요소의 값이
> -2~2 사이일 확률은 95%이다. 초깃값이 평균 0, 표준편차 10인 정규분포를 따르도록 가중치 초기화를 바꾸면
> 학습 결과가 어떻게 될까? 초기화 방법을 바꿔 서로 다른 초깃값으로 10번 이상 반복 학습한 후,
> 학습한 모델이 네 샘플을 모두 바르게 분류하는지 확인해 보자.
>
> 힌트: 초깃값이 크면 가중합도 커진다. [그림 2-4]에서 입력이 큰 구간의 시그모이드 함수 기울기를 살펴보자.

## 연습 문제 2-5

> OR 게이트와 NAND 게이트를 시뮬레이션하는 모델을 각각 만들어 보자. OR 게이트는 두 개의 입력이 모두 0일 때만
> 출력이 0이 되고, NAND 게이트는 두 입력이 모두 1일 때만 출력이 0이 된다.

In [4]:
GATES = {
    'OR': torch.tensor([[0.], [1.], [1.], [1.]]),      # 둘 다 0일 때만 0
    'NAND': torch.tensor([[1.], [1.], [1.], [0.]]),    # 둘 다 1일 때만 0
}

for name, Y_true in GATES.items():
    torch.manual_seed(SEED)
    weights = torch.randn((2, 1), requires_grad=True)
    bias = torch.randn((1,), requires_grad=True)
    loss = train_quiet(X, Y_true, weights, bias)
    pred = classify(X, weights, bias)
    print(f'[{name} 게이트] 손실 {loss:.4f}, 분류 성공 {torch.equal(pred, Y_true)}')
    print(f'    가중치 ({weights[0].item():.2f}, {weights[1].item():.2f}), 편향 {bias.item():.2f}')
    print(f'    결정 경계: 기울기 {-weights[0].item() / weights[1].item():.2f}, x2축 절편 {-bias.item() / weights[1].item():.2f}')

[OR 게이트] 손실 0.0360, 분류 성공 True
    가중치 (2.50, 2.44), 편향 -0.87
    결정 경계: 기울기 -1.02, x2축 절편 0.36


[NAND 게이트] 손실 0.0611, 분류 성공 True
    가중치 (-1.76, -1.78), 편향 2.82
    결정 경계: 기울기 -0.99, x2축 절편 1.58


### 풀이 해설

바뀌는 것은 정답 텐서 `Y_true` 하나뿐이다. 모델도, 손실 함수도, 학습 루프도 AND 게이트와 똑같다.
본문 p7이 말한 "문제가 달라져도 구조는 바뀌지 않는다"를 가장 적은 수고로 확인할 수 있는 문제다.

두 게이트의 결정 경계를 비교하면 성격이 드러난다. OR 게이트는 (0, 0) 하나만 떼어 내야 하므로 경계가 원점 쪽으로
당겨지고, NAND 게이트는 (1, 1) 하나만 떼어 내되 **그쪽이 0**이어야 하므로 가중치의 부호가 AND와 반대가 된다.
즉 NAND는 AND의 결정 경계를 그대로 두고 어느 쪽을 1로 볼지만 뒤집은 것과 같다.

### 문제 검토

- **적절성: 적합.** AND 게이트 예제에서 정답 텐서만 바꾸면 되므로, 퍼셉트론이 '구조는 그대로 두고 파라미터만
  바꿔 다른 문제를 푼다'는 성질을 손으로 확인하게 한다. 다음 문제 2-6(입력 3개)과 함께 난도도 완만하게 올라간다.
- **[중요] ★ 힌트가 지문과 똑같다.** 지문 마지막 문장과 힌트가 조사만 다르고 내용이 같다.

  > 지문: OR 게이트는 두 개의 입력이 모두 0일 때만 출력이 0이 **되고**, NAND 게이트는 두 입력이 모두 1일 때만 출력이 0이 **된다**.
  >
  > 힌트: OR 게이트는 두 개의 입력이 모두 0일 때만 출력이 0**이고**, NAND 게이트는 두 입력이 모두 1일 때만 출력이 0**이다**.

  편집 과정에서 같은 문장이 두 번 들어간 것으로 보인다. 힌트를 지우거나, 실제로 도움이 되는 내용으로 바꿔야 한다.
  풀이 과정에서 보면 독자가 실제로 막히는 지점은 게이트의 정의가 아니라 **연산표를 정답 텐서로 옮기는 부분**이다.

**윤문안**

> **2-5**. OR 게이트와 NAND 게이트를 시뮬레이션하는 모델을 각각 만들어 보자.
> OR 게이트는 두 개의 입력이 모두 0일 때만 출력이 0이 되고, NAND 게이트는 두 입력이 모두 1일 때만 출력이 0이 된다.
>
> 힌트: 입력 텐서 `X`와 학습 루프는 AND 게이트 예제를 그대로 쓰고, 정답 텐서 `Y_true`만 각 게이트의 연산표에 맞게 바꾸면 된다.

## 연습 문제 2-6

> 입력이 세 개인 AND 게이트를 시뮬레이션하는 모델을 만들어 보자. 이 게이트는 세 입력이 모두 1일 때만 1을 출력한다.

In [5]:
# 입력이 세 개이므로 샘플은 2^3 = 8개
X3 = torch.tensor([[0., 0., 0.], [0., 0., 1.], [0., 1., 0.], [0., 1., 1.],
                   [1., 0., 0.], [1., 0., 1.], [1., 1., 0.], [1., 1., 1.]])
Y3 = torch.tensor([[0.], [0.], [0.], [0.], [0.], [0.], [0.], [1.]])

def run_and3(epochs, learning_rate, seed=SEED):
    torch.manual_seed(seed)
    # 입력이 세 개이므로 가중치는 (3, 1) 형태
    weights = torch.randn((3, 1), requires_grad=True)
    bias = torch.randn((1,), requires_grad=True)
    loss = train_quiet(X3, Y3, weights, bias, epochs=epochs, learning_rate=learning_rate)
    ok = torch.equal(classify(X3, weights, bias), Y3)
    return loss, ok, weights, bias

# 먼저 본문 2-2절과 같은 하이퍼파라미터로 학습해 본다
loss, ok, weights3, bias3 = run_and3(EPOCHS, LR)
print(f'학습률 {LR}, {EPOCHS}에포크 -> 손실 {loss:.4f}, 분류 성공 {ok}')
print(f'    가중치 {[round(w, 2) for w in weights3.flatten().tolist()]}, 편향 {bias3.item():.2f}')
with torch.no_grad():
    print(f'    (1, 1, 1)의 가중합: {(torch.tensor([[1., 1., 1.]]) @ weights3 + bias3).item():.3f}')

학습률 0.1, 1000에포크 -> 손실 0.0692, 분류 성공 False
    가중치 [0.91, 0.83, 0.82], 편향 -2.79
    (1, 1, 1)의 가중합: -0.221


In [6]:
# 에포크와 학습률을 바꿔 가며 확인
print(f'{"학습률":>6} {"에포크":>7} {"손실":>9} {"분류 성공":>9}')
print('-' * 38)
for epochs, learning_rate in ((1000, 0.1), (2000, 0.1), (5000, 0.1), (1000, 0.3), (1000, 0.5)):
    loss, ok, _, _ = run_and3(epochs, learning_rate)
    print(f'{learning_rate:6.1f} {epochs:7d} {loss:9.4f} {str(ok):>9}')

# 시드를 바꿔도 본문 설정으로는 실패하는지 확인
results = [run_and3(EPOCHS, LR, seed=s)[1] for s in range(1, 11)]
print(f'\n학습률 {LR}, {EPOCHS}에포크로 시드 1~10을 시도한 결과: 성공 {sum(results)}회 / 10회')

   학습률     에포크        손실     분류 성공
--------------------------------------


   0.1    1000    0.0692     False


   0.1    2000    0.0485      True


   0.1    5000    0.0256      True
   0.3    1000    0.0375      True


   0.5    1000    0.0256      True



학습률 0.1, 1000에포크로 시드 1~10을 시도한 결과: 성공 0회 / 10회


위 결과에서 **학습률 0.1, 1,000에포크로는 어떤 시드로 시작해도 실패**한다는 것을 알 수 있다.
에포크를 2,000으로 늘리거나 학습률을 0.3으로 올리면 여덟 샘플을 모두 바르게 분류한다.

샘플이 네 개에서 여덟 개로 늘었는데 정답이 1인 샘플은 여전히 하나뿐이다.
손실은 여덟 샘플의 평균이므로, 정답이 1인 그 하나를 맞히라고 파라미터를 미는 힘이 절반으로 약해진 셈이다.
입력이 늘면 같은 하이퍼파라미터로는 학습이 더뎌진다는 것을 보여 주는 예다.

### 풀이 해설

입력이 하나 늘었을 때 코드에서 바뀌는 것은 두 가지뿐이다. 데이터가 4개에서 8개로 늘고(2³),
가중치 텐서가 `(2, 1)`에서 `(3, 1)`로 바뀐다. 모델 함수 `X @ weights + bias`는 한 글자도 고칠 필요가 없다.
행렬곱이 입력 개수에 맞춰 알아서 동작하기 때문인데, 본문 p9가 행렬곱을 강조한 이유가 여기서 드러난다.

**그런데 본문과 같은 하이퍼파라미터로는 학습에 실패한다.** 학습률 0.1, 1,000에포크로 돌리면 손실은 0.0692로
작아 보이지만, 정작 (1, 1, 1)의 가중합이 음수라서 여덟 샘플 중 그 하나를 틀린다. 아래에서 원인을 확인한다.

### 문제 검토

- **적절성: 적합.** 코드를 거의 고치지 않고도 입력 차원을 늘릴 수 있다는 것을 확인시켜,
  본문의 행렬곱 설명과 파이토치가 텐서를 기본 자료형으로 삼는 이유를 실감하게 한다.
  2-3절 연습 문제 2-8에서 이 모델을 고수준 API로 다시 만들므로 연결도 좋다.
- **[중요] ★★ 본문의 하이퍼파라미터로는 풀리지 않는다.** 아래 실험에서 보듯 학습률 0.1, 1,000에포크로는
  **시도한 열 개 시드 모두 실패**한다. 문제는 손실이 0.0692로 작아 보인다는 점이다. AND 게이트 예제의
  최종 손실이 0.0576이었으므로 독자는 비슷한 값을 보고 성공했다고 판단하기 쉽다.
  분류 결과를 따로 확인하지 않으면 틀린 모델을 맞다고 여기고 넘어간다.
  에포크를 2,000으로 늘리거나 학습률을 0.3으로 올리면 해결된다.
- **원인은 문제 자체에 있다.** 양성 샘플이 네 개 중 하나에서 여덟 개 중 하나로 줄어, 정답이 1인 방향으로
  파라미터를 미는 힘이 절반으로 약해졌다. 입력이 늘면 같은 설정으로는 학습이 더뎌진다는 뜻이다.
  이는 알아 둘 가치가 있는 성질이므로, 지문이 학습이 끝난 뒤 분류 결과를 확인하도록 유도하면
  독자가 스스로 발견하고 하이퍼파라미터를 조정하게 된다.
- **[검토] 결정 경계가 평면이 된다는 점을 짚어 주면 좋다.** 2-1절 내내 결정 경계를 '직선'으로 설명했는데
  입력이 셋이 되면 평면이 된다. 이 문제가 그 전환을 겪는 첫 자리다.

**윤문안**

> **2-6**. 입력이 세 개인 AND 게이트를 시뮬레이션하는 모델을 만들어 보자. 이 게이트는 세 입력이 모두 1일 때만
> 1을 출력한다. 학습을 마친 뒤에는 손실뿐 아니라 여덟 샘플의 분류 결과가 모두 맞는지 확인하고,
> 맞지 않는다면 하이퍼파라미터를 조정해 보자. 이 모델의 결정 경계가 어떤 모양일지도 함께 생각해 보자.